# RTX-OOM-Guard — Colab T4 Validation

**Goal:** Measure whether proactive defragmentation delays or prevents OOM during a Transformer training loop with deliberate memory fragmentation.

**Hardware:** NVIDIA T4 (free Colab), 15.6GB VRAM  
**Runtime:** GPU → T4  
**Expected time:** ~5 minutes

## Instructions
1. Open in Colab (Runtime → Change runtime type → T4 GPU)
2. Run all cells in order
3. Results saved to `results/colab_t4_results.json`

In [ ]:
# Cell 1: Install rtx_oom_guard from GitHub
!pip install -q git+https://github.com/poojakira/RTX-OOM-Guard.git

import sys
print(f"Python {sys.version}")

import torch
print(f"PyTorch {torch.__version__}")

import rtx_oom_guard
print(f"rtx_oom_guard {rtx_oom_guard.__version__} installed successfully")

assert torch.cuda.is_available(), "NO GPU — set Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 2: Imports and config
import torch
import gc
import json
import time
import random
from rtx_oom_guard import auto_instrument

# Workload sized to stress T4 (15.6GB)
# Model params + optimizer state + activations should approach ~13-14GB
D_MODEL = 1024
NHEAD = 16
NUM_LAYERS = 12
BATCH = 16
SEQ_LEN = 256
MAX_STEPS = 50

print(f"Workload: d_model={D_MODEL}, nhead={NHEAD}, layers={NUM_LAYERS}")
print(f"Batch: {BATCH} x seq_len={SEQ_LEN}")
print(f"Max steps: {MAX_STEPS}")

In [ ]:
# Cell 3: Fragmentation helper
def fragment_memory(n_tensors=300, sizes_mb=(1, 2, 4, 8, 16, 32, 64)):
    """Allocate and free tensors of varying sizes to fragment CUDA caching allocator.
    
    Strategy: allocate many tensors of random sizes, then free ~60% of them
    in a scattered pattern. The remaining tensors pin holes in the allocator,
    preventing large contiguous allocations.
    """
    tensors = []
    for _ in range(n_tensors):
        size_mb = random.choice(sizes_mb)
        t = torch.randn(size_mb * 256 * 1024, device='cuda')  # 256K floats = 1MB
        tensors.append(t)
    
    # Free ~60% in scattered pattern (every 2nd and 3rd)
    keep = []
    for i, t in enumerate(tensors):
        if i % 5 in (0, 3):  # keep 40%
            keep.append(t)
    
    del tensors
    gc.collect()
    torch.cuda.empty_cache()
    
    stats = torch.cuda.memory_stats()
    allocated = stats['allocated_bytes.all.current'] / 1e9
    reserved = stats['reserved_bytes.all.current'] / 1e9
    frag_ratio = 1 - (allocated / reserved) if reserved > 0 else 0
    print(f"  Fragmentation: allocated={allocated:.2f}GB, reserved={reserved:.2f}GB, frag_ratio={frag_ratio:.2%}")
    return keep

print("fragment_memory() defined.")

In [ ]:
# Cell 4: Baseline — train WITHOUT guard
def train_baseline():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    model = torch.nn.Transformer(
        d_model=D_MODEL, nhead=NHEAD,
        num_encoder_layers=NUM_LAYERS,
        num_decoder_layers=NUM_LAYERS,
        batch_first=True
    ).cuda()
    optimizer = torch.optim.Adam(model.parameters())
    
    print("  Fragmenting memory...")
    frags = fragment_memory()
    
    steps_completed = 0
    t0 = time.time()
    try:
        for step in range(MAX_STEPS):
            src = torch.randn(BATCH, SEQ_LEN, D_MODEL, device='cuda')
            tgt = torch.randn(BATCH, SEQ_LEN, D_MODEL, device='cuda')
            out = model(src, tgt)
            loss = out.sum()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            steps_completed = step + 1
            del src, tgt, out, loss
            if step % 10 == 0:
                mem = torch.cuda.memory_allocated() / 1e9
                print(f"    step {step}: {mem:.2f}GB allocated")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"  OOM at step {steps_completed}!")
        else:
            raise
    
    elapsed = time.time() - t0
    stats = torch.cuda.memory_stats()
    result = {
        "steps": steps_completed,
        "peak_gb": round(stats['allocated_bytes.all.peak'] / 1e9, 3),
        "retries": stats.get('num_alloc_retries', 0),
        "oom": steps_completed < MAX_STEPS,
        "elapsed_s": round(elapsed, 1)
    }
    
    del model, optimizer, frags
    gc.collect()
    torch.cuda.empty_cache()
    return result

print("=== BASELINE (no guard) ===")
baseline = train_baseline()
print(f"Result: {json.dumps(baseline, indent=2)}")

In [ ]:
# Cell 5: WITH rtx_oom_guard
def train_with_guard():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    
    model = torch.nn.Transformer(
        d_model=D_MODEL, nhead=NHEAD,
        num_encoder_layers=NUM_LAYERS,
        num_decoder_layers=NUM_LAYERS,
        batch_first=True
    ).cuda()
    optimizer = torch.optim.Adam(model.parameters())
    
    # Instrument with rtx_oom_guard
    model, optimizer = auto_instrument(model, optimizer)
    
    print("  Fragmenting memory...")
    frags = fragment_memory()
    
    steps_completed = 0
    t0 = time.time()
    try:
        for step in range(MAX_STEPS):
            src = torch.randn(BATCH, SEQ_LEN, D_MODEL, device='cuda')
            tgt = torch.randn(BATCH, SEQ_LEN, D_MODEL, device='cuda')
            out = model(src, tgt)
            loss = out.sum()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            steps_completed = step + 1
            del src, tgt, out, loss
            if step % 10 == 0:
                mem = torch.cuda.memory_allocated() / 1e9
                print(f"    step {step}: {mem:.2f}GB allocated")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"  OOM at step {steps_completed}!")
        else:
            raise
    
    elapsed = time.time() - t0
    stats = torch.cuda.memory_stats()
    result = {
        "steps": steps_completed,
        "peak_gb": round(stats['allocated_bytes.all.peak'] / 1e9, 3),
        "retries": stats.get('num_alloc_retries', 0),
        "oom": steps_completed < MAX_STEPS,
        "elapsed_s": round(elapsed, 1)
    }
    
    del model, optimizer, frags
    gc.collect()
    torch.cuda.empty_cache()
    return result

print("\n=== WITH RTX-OOM-GUARD ===")
guarded = train_with_guard()
print(f"Result: {json.dumps(guarded, indent=2)}")

In [ ]:
# Cell 6: Summary and save
results = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_gb": round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1),
    "workload": {
        "d_model": D_MODEL,
        "nhead": NHEAD,
        "num_layers": NUM_LAYERS,
        "batch_size": BATCH,
        "seq_len": SEQ_LEN,
        "max_steps": MAX_STEPS
    },
    "baseline": baseline,
    "guarded": guarded,
    "improvement": {
        "extra_steps": guarded["steps"] - baseline["steps"],
        "peak_reduction_gb": round(baseline["peak_gb"] - guarded["peak_gb"], 3),
        "oom_prevented": baseline["oom"] and not guarded["oom"]
    }
}

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(json.dumps(results, indent=2))

# Save
import os
os.makedirs('results', exist_ok=True)
with open('results/colab_t4_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved to results/colab_t4_results.json")

## Interpreting Results

| Outcome | What it means |
|---------|---------------|
| Guard prevents OOM | Compaction freed enough fragmented blocks to survive |
| Guard delays OOM (more steps) | Partial win — compaction helps but optimizer state still fragments |
| No OOM in either run | Workload didn't hit memory ceiling — fragmentation wasn't severe enough |
| Guard doesn't help | Optimizer state (Adam exp_avg/exp_avg_sq) dominates fragmentation and isn't compacted |

**Any result is valid.** A documented negative result with root-cause analysis is engineering. Fabricated positive results are fraud.